# Story Writing Benchmark - 5. Calibration Robustness

## Setup

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path("../../scripts").resolve()))
from pareto_figures import (
    DATASETS, PROJECT_ROOT, bootstrap_calibration, compute_cat_value,
    load_dataset, metric_spec, operating_point_at_threshold,
)

sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["figure.dpi"] = 120

SCORE_COL = {
    "mae":     "ucb_mae",
    "rmse":    "ucb_rmse",
    "rho":     "lcb_rho",
    "kendall": "lcb_kendall",
}
METRIC_LABEL = {
    "mae":     "MAE",
    "rmse":    "RMSE",
    "rho":     "Spearman \u03c1",
    "kendall": "Kendall \u03c4",
}
K_PALETTE = {1: "#E63946", 2: "#F4A261", 5: "#2E86AB", 10: "#3B1F2B", 50: "#A23B72"}

K_VALUES = [1, 2, 5, 10]
N_SEEDS  = 50
ALPHA    = 0.05
METRICS  = ["mae", "rmse", "rho", "kendall"]
FIGURES_DIR = PROJECT_ROOT / "scripts" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
cfg = next(c for c in DATASETS if c.name == "story_writing_benchmark")
pairs, errors, evaluator_names = load_dataset(cfg)

per_q_min = (errors.groupby(cfg.id_keys + ["category"])["abs_err"].min().reset_index())
n_per_cat = len(per_q_min) // len(cfg.categories)
print(f"Dataset : {cfg.label}")
print(f"Rubrics : {len(cfg.categories)}  |  n_per_cat: {n_per_cat}")
print(f"Evaluators ({len(evaluator_names)}): {evaluator_names}")

## Run bootstraps

For each `k`, draw `N_SEEDS` independent bootstrap calibrations with `m = ⌊√n⌋` fixed. Each call produces one (rubric × evaluator) calibrated-score table.

In [ ]:
print(f"Running {N_SEEDS} seeds × {len(K_VALUES)} k-values = {N_SEEDS * len(K_VALUES)} bootstrap calls ...")
cal_results = {
    k: [bootstrap_calibration(pairs, k=k, alpha=ALPHA, seed=s) for s in range(N_SEEDS)]
    for k in K_VALUES
}
print("Done.")

## Calibration robustness figures

For each metric, plot the distribution of the no-fallback realized quality across seeds as a violin per k. Lower spread = more stable calibration.

In [ ]:
def per_call_realized_metric(cal_list, cat_value, categories, evaluator_names,
                              score_col, higher_is_better, n_per_cat):
    """Realized macro-quality under no-fallback policy for each bootstrap call."""
    no_fallback = float("-inf") if higher_is_better else float("inf")
    means = []
    for cal in cal_list:
        score_table = (
            cal.pivot_table(index="category", columns="evaluator", values=score_col)
               .reindex(index=categories, columns=evaluator_names)
        )
        op = operating_point_at_threshold(
            score_table, cat_value, categories,
            0, n_per_cat, higher_is_better, no_fallback,
        )
        means.append(op["metric"])
    return means


def make_figure(metric, cal_results, k_values, n_seeds, cat_value, n_per_cat, cfg):
    spec = metric_spec(metric)
    score_col        = spec["score_col"]
    higher_is_better = spec["higher_is_better"]

    rows = []
    for k in k_values:
        means = per_call_realized_metric(
            cal_results[k], cat_value, cfg.categories, evaluator_names,
            score_col, higher_is_better, n_per_cat,
        )
        for s, v in enumerate(means):
            rows.append({"k": k, "seed": s, "score": v})
    df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(4.5, 4))
    palette_dict = [K_PALETTE.get(k, "#888888") for k in k_values]
    sns.violinplot(
        data=df, x="k", y="score", order=k_values,
        hue="k", hue_order=k_values, palette=palette_dict, legend=False,
        inner="quartile", cut=0, density_norm="width",
        ax=ax, linewidth=1.0,
    )
    sns.stripplot(
        data=df, x="k", y="score", order=k_values,
        size=1.8, color="#222222", alpha=0.25, jitter=0.18, ax=ax,
    )
    for i, k in enumerate(k_values):
        sub = df[df["k"] == k]["score"]
        if len(sub) == 0:
            continue
        med = float(sub.median())
        ax.text(i, med, f"  med={med:.3f}", va="center", ha="left",
                fontsize=8, color="black",
                bbox=dict(facecolor="white", edgecolor="none", pad=1.5, alpha=0.7))
    ax.set_xlabel("k (# of repeated \u230a\u221an\u230b sampling)")
    ax.set_ylabel(
        f"{METRIC_LABEL[metric]} under no-fallback\n"
        f"(one value per calibration; {n_seeds} seeds per k)"
    )
    ax.set_title(
        f"{cfg.label} \u2014 {METRIC_LABEL[metric]}\n(m = \u230a\u221an\u230b fixed)",
        fontweight="bold",
    )
    ax.grid(True, axis="y", alpha=0.3, linewidth=0.6)
    ax.grid(False, axis="x")
    plt.tight_layout()
    return fig

In [ ]:
for metric in METRICS:
    cat_value = compute_cat_value(pairs, errors, metric, cfg.categories, evaluator_names)
    fig = make_figure(metric, cal_results, K_VALUES, N_SEEDS, cat_value, n_per_cat, cfg)
    base = FIGURES_DIR / f"calibration_robustness_{cfg.name}_{metric}"
    fig.savefig(f"{base}.pdf", bbox_inches="tight")
    fig.savefig(f"{base}.png", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"saved {base.name}.pdf / .png")